
# 練習問題: 2D 拡散方程式 (インクの広がり) を陽解法で解く

## 目標

水中のインクが時間とともに広がる様子 (拡散方程式 `u_t = D(u_xx + u_yy)`) を, 差分法 (陽解法 FTCS) で計算する。
B1・B2 と同じ stencil の構造で, 空間の二重ループを `collapse(2)` で並列化する。**反射境界で総量が保存される**ことを検算に使う。

## 課題

`diffusion.cpp` (または `diffusion.f90`) は, L×L の領域の中央にインクの塊 (濃度1) を置き, 各ステップで全格子点を

```
u^{n+1} = u + alpha × (上 + 下 + 左 + 右 − 4×中央)
```

で更新する。`alpha = D·Δt/Δx²` で安定条件は `alpha ≤ 0.25` (このコードでは 0.2)。境界は「反射 (断熱)」: 領域の外の隣は自分自身とみなす (端で添字を留める) ので, インクは外へ漏れない。

`TODO` の箇所に **OpenMP の指示を追加** し, 更新の二重ループを並列化せよ。

- C++: 二重ループの直前に `#pragma omp parallel for collapse(2)` を1行加える。
- Fortran: 二重ループを `!$omp parallel do collapse(2)` と `!$omp end parallel do` で囲む。

ポイント:

- 並列化するのは**空間ループ** (各時間ステップ内)。時間ループは逐次 (B2 と同じ)。
- 各点の更新は独立で総和も無いので reduction は不要 (B1 の残差判定とは違う)。
- 過去・未来の2枚の配列をポインタで入れ替えて使う。

## コンパイルと実行

```
# C++
nvc++ -fast -mp=multicore diffusion.cpp -o diffusion.exe

# Fortran
nvfortran -fast -mp=multicore diffusion.f90 -o diffusion.exe
```

第1引数で格子サイズ `L`, 第2引数で時間ステップ数。

```
OMP_NUM_THREADS=4 ./diffusion.exe 256 500
```

## 期待される結果

```
L=256, steps=500: 総量 1024.000000 -> 1024.000000 (保存されるはず), 最大濃度 0.550256
```

- **総量保存**: 反射境界なのでインクは漏れず, 総量 (全セルの和) は時間が経っても一定 (丸め誤差程度しか変わらない) はず。これが正しさの証拠になる。
- **最大濃度の低下**: インクが広がるほど, 中央のピーク濃度は下がる。ステップ数を増やすと, より平らに (一様に) 近づく。
- スレッド数を変えても結果は変わらない (各点の更新が独立な決定的計算なので)。
- (発展) `alpha` を 0.3 など 0.25 超にすると不安定になり, 値が振動・発散することも見てみよ。



# 1. ツールの読み込み
- AIチュータ及びジョブ投入ツールの読み込み (カーネル起動後に一度実行すればよい)
  - `heytutor` : `%%hey` でAIチュータに質問できるようになる (使い方は末尾を参照)
  - `wisteria_submit` : `%%bash_submit` (先頭に `#PJM ...` を書く) でジョブ投入できるようになる


In [ ]:
import heytutor
import wisteria_submit

# 2. C++ ベースコード

In [ ]:
%%writefile_ diffusion.cpp
#include <cstdio>
#include <cstdlib>
#include <omp.h>

/* 2D 拡散方程式 u_t = D (u_xx + u_yy) を陽解法 (FTCS) で時間発展させる。
   中央に置いたインクの塊が時間とともに広がる様子を計算する。
   更新式 (5点ラプラシアン, alpha = D*dt/dx^2, 安定条件 alpha <= 0.25):
     u^{n+1}[i][j] = u + alpha * (上 + 下 + 左 + 右 - 4*中央)
   境界は「反射 (断熱)」: 領域外の隣は自分自身とみなす (端の添字をはみ出さないよう留める)。
   → インクが外へ漏れないので, 全体の総量は時間によらず保存される (検算に使う)。 */
int main(int argc, char ** argv) {
  int    L     = (argc > 1 ? atoi(argv[1]) : 256);     /* 一辺の格子点数 */
  int    steps = (argc > 2 ? atoi(argv[2]) : 500);     /* 時間ステップ数 */
  double alpha = 0.2;                                  /* D*dt/dx^2, 安定 */

  int n = L * L;
  double * u  = (double *)calloc(n, sizeof(double));
  double * un = (double *)calloc(n, sizeof(double));
  /* 初期条件: 中央の正方形ブロックに濃度 1 のインクを置く。 */
  int lo = L / 2 - L / 16, hi = L / 2 + L / 16;
  for (int i = lo; i < hi; i++)
    for (int j = lo; j < hi; j++)
      u[i * L + j] = 1.0;
  double mass0 = 0.0;
  for (int k = 0; k < n; k++) mass0 += u[k];

  double t0 = omp_get_wtime();
  for (int t = 0; t < steps; t++) {
    /* 全格子点を更新 (時間1ステップ進める)。端では添字を留めて反射境界にする。 */
    // TODO: 更新の二重ループを #pragma omp parallel for collapse(2) で並列化せよ.
    for (int i = 0; i < L; i++) {
      for (int j = 0; j < L; j++) {
        int im = (i > 0 ? i - 1 : i), ip = (i < L - 1 ? i + 1 : i);
        int jm = (j > 0 ? j - 1 : j), jp = (j < L - 1 ? j + 1 : j);
        double c = u[i * L + j];
        un[i * L + j] = c + alpha * (u[im * L + j] + u[ip * L + j]
                                   + u[i * L + jm] + u[i * L + jp] - 4.0 * c);
      }
    }
    double * tmp = u; u = un; un = tmp;
  }
  double elapsed = omp_get_wtime() - t0;

  /* 検算: 総量 (sum) が保存されているか。最大濃度は広がるほど下がる。 */
  double mass1 = 0.0, maxc = 0.0;
  for (int k = 0; k < n; k++) { mass1 += u[k]; if (u[k] > maxc) maxc = u[k]; }
  printf("L=%d, steps=%d: 総量 %.6f -> %.6f (保存されるはず), 最大濃度 %.6f\n",
         L, steps, mass0, mass1, maxc);
  printf("elapsed = %.3f sec\n", elapsed);
  free(u); free(un);
  return 0;
}

## 2-1. コンパイル

In [ ]:
%%bash_
PATH=/work/opt/local/x86_64/cores/nvidia/23.3/Linux_x86_64/23.3/compilers/bin:/opt/FJSVxtclanga/tcsds-1.2.41/bin:$PATH
nvc++ -fast -mp=multicore diffusion.cpp -o diffusion_cpp.exe

## 2-2. 実行
- 計算ノードにジョブとして投入して実行する。スレッド数・キュー・制限時間は `#PJM` 行で調整する。
- すぐにログインノードで試したいときは, 先頭の `%%bash_submit` を `%%bash_` に書き換えて (`#PJM` 行はコメントなので無視される) 実行すればよい。ただし数秒で終わる軽いジョブに限る。

In [ ]:
%%bash_submit
#PJM -L rscgrp=lecture-a
#PJM -L elapse=0:01:00
#PJM -L gpu=1
#PJM --omp thread=9
#PJM -g gt69
#PJM -j
#PJM -o 0output.txt

./diffusion_cpp.exe

# 3. Fortran ベースコード

In [ ]:
%%writefile_ diffusion.f90
! 2D 拡散方程式 u_t = D (u_xx + u_yy) を陽解法 (FTCS) で時間発展させる。
! 中央に置いたインクの塊が時間とともに広がる様子を計算する。
! 更新式 (5点ラプラシアン, alpha=D*dt/dx^2, 安定条件 alpha<=0.25):
!   u^{n+1}(i,j) = u + alpha*(上+下+左+右 - 4*中央)
! 境界は「反射(断熱)」: 領域外の隣は自分自身とみなす (端で添字を留める)。
! → インクが外へ漏れないので, 全体の総量は時間によらず保存される (検算に使う)。
program diffusion
  use omp_lib
  implicit none
  integer :: L, steps, t, i, j
  real(8) :: alpha, mass0, mass1, maxc, t0, elapsed
  real(8), allocatable, target :: arr1(:,:), arr2(:,:)
  real(8), pointer :: u(:,:), un(:,:), tmp(:,:)
  integer :: lo, hi
  character(len=32) :: arg
  L = 256; steps = 500; alpha = 0.2d0
  if (command_argument_count() >= 1) then
     call get_command_argument(1, arg); read (arg, *) L
  end if
  if (command_argument_count() >= 2) then
     call get_command_argument(2, arg); read (arg, *) steps
  end if

  allocate(arr1(0:L-1,0:L-1), arr2(0:L-1,0:L-1))
  arr1 = 0.0d0; arr2 = 0.0d0
  u => arr1; un => arr2
  ! 初期条件: 中央の正方形ブロックに濃度 1 のインクを置く。
  lo = L/2 - L/16; hi = L/2 + L/16
  do j = lo, hi - 1
     do i = lo, hi - 1
        u(i,j) = 1.0d0
     end do
  end do
  mass0 = sum(u)

  t0 = omp_get_wtime()
  do t = 1, steps
     ! 全格子点を更新 (時間1ステップ進める)。端では添字を留めて反射境界にする。
     ! TODO: 更新の二重ループを !$omp parallel do collapse(2) で並列化せよ.
     do j = 0, L - 1
        do i = 0, L - 1
           un(i,j) = u(i,j) + alpha * ( &
                u(max(i-1,0),   j) + u(min(i+1,L-1), j) + &
                u(i, max(j-1,0))   + u(i, min(j+1,L-1)) - 4.0d0 * u(i,j) )
        end do
     end do
     ! TODO: 上で始めた parallel do 領域を閉じる (!$omp end parallel do).
     tmp => u; u => un; un => tmp
  end do
  elapsed = omp_get_wtime() - t0

  ! 検算: 総量 (sum) が保存されているか。最大濃度は広がるほど下がる。
  mass1 = sum(u)
  maxc = maxval(u)
  print "(a,i0,a,i0,a,f0.6,a,f0.6,a,f0.6)", &
       "L=", L, ", steps=", steps, ": 総量 ", mass0, " -> ", mass1, &
       " (保存されるはず), 最大濃度 ", maxc
  print "(a,f0.3,a)", "elapsed = ", elapsed, " sec"
end program diffusion

## 3-1. コンパイル

In [ ]:
%%bash_
PATH=/work/opt/local/x86_64/cores/nvidia/23.3/Linux_x86_64/23.3/compilers/bin:/opt/FJSVxtclanga/tcsds-1.2.41/bin:$PATH
nvfortran -fast -mp=multicore diffusion.f90 -o diffusion_f90.exe

## 3-2. 実行
- 計算ノードにジョブとして投入して実行する。スレッド数・キュー・制限時間は `#PJM` 行で調整する。
- すぐにログインノードで試したいときは, 先頭の `%%bash_submit` を `%%bash_` に書き換えて (`#PJM` 行はコメントなので無視される) 実行すればよい。ただし数秒で終わる軽いジョブに限る。

In [ ]:
%%bash_submit
#PJM -L rscgrp=lecture-a
#PJM -L elapse=0:01:00
#PJM -L gpu=1
#PJM --omp thread=9
#PJM -g gt69
#PJM -j
#PJM -o 0output.txt

./diffusion_f90.exe


# 4. 発展目標 (できる範囲で挑戦)
- この問題の基本は **マルチコア並列化** (`#pragma omp parallel for` / `!$omp parallel do` など)。まずはここまでを目指す。
- 余力があれば次にも挑戦:
  - **GPUで並列化**: コンパイルを `-mp=gpu` にして, 重いループに `#pragma omp target teams distribute parallel for` (+ 必要に応じて `map`) を付ける。
  - **SIMD化** (11/12章): 内側ループに `#pragma omp simd`, またはベクトル型。 **ILP向上** (13章): ベクトル長 `nl` の調整。
- どこまで速くできるか挑戦し, その効果を下の「性能を比べる」で可視化しよう。

# 5. 性能を比べる (任意)
- 各プログラムは主計算部分の所要時間を `elapsed = ... sec` の形で表示する。
- 構成を変えて (スレッド数, SIMDの有無, GPU など) 実行し, 得られた **経過秒** を下の `DATA` に「ラベルと秒」で並べて実行すると, 棒グラフと「基準(先頭)比のスピードアップ」が出る。
- 試した構成だけ書けばよい (`#` で始まる行は無視)。


In [ ]:
import matplotlib.pyplot as plt

# 各構成の (ラベル, 経過秒) を並べる。先頭の行を基準(スピードアップ=1)にする。
# 自分が実際に試した構成の数値に書き換える。
DATA = [
    ("serial",    1.00),
    ("omp-8",     0.20),
    # ("simd",      0.50),
    # ("simd+omp",  0.07),
    # ("gpu",       0.05),
]

labels = [d[0] for d in DATA]
secs   = [float(d[1]) for d in DATA]
speed  = [secs[0] / s for s in secs]                 # 先頭(基準)比のスピードアップ
fig, ax = plt.subplots(1, 2, figsize=(9, 3))
ax[0].bar(labels, secs)
ax[0].set_ylabel("elapsed (s)")
ax[0].set_title("time (lower = faster)")
ax[1].bar(labels, speed)
ax[1].set_ylabel("speedup vs " + labels[0])
ax[1].set_title("speedup (higher = faster)")
for a in ax:
    a.tick_params(axis="x", rotation=30)
plt.tight_layout()
plt.show()


# 6. AIチュータへの質問の仕方 (参考)
- 先頭で `import heytutor` 済みなら, セルに `%%hey` と書いて質問できる。
- ChatGPTなどと同様に自由に質問してよい。ただし「答えをそのまま教えて」などは自制すること。
- セル内で使える変数 (自動で展開される):
  - `{file:FILENAME}` : _FILENAME_ の中身 (例: `{file:problem.md}`, `{file:diffusion.cpp}`)
  - `{bash[-1]}` : 最後に実行した `%%bash_` セルの入力・出力, `{bash[-2]}` = その前, ...
- 以下は質問例 (必要に応じてコピーして使う。Fortranなら `.cpp` を `.f90` に書き換える)。

## 6-1. 一般的な質問

In [ ]:
%%hey

C++の関数定義の文法どんなだっけ?

## 6-2. この問題に関するヒント

In [ ]:
%%hey

この問題に関するヒントを教えて

問題:
{file:problem.md}

## 6-3. 困ったときのヘルプ
- コンパイル時や実行時のエラー直後に実行するとエラーに関するヘルプが得られる。

In [ ]:
%%hey

以下のエラーが出た。何が間違い?

プログラム:
{file:diffusion.cpp}

コマンドと実行結果:
{bash[-1]}

## 6-4. フィードバック
- 答えが出た後も, 無駄なところやより良いやり方がないかを聞くことを推奨。

In [ ]:
%%hey

私の答に対するフィードバックをください。

問題:
{file:problem.md}

私の答:
{file:diffusion.cpp}